In [1]:
import os, time
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier.constants import *
import tensorflow as tf
from cnnClassifier.utils.common import read_yaml, create_directories

I0000 00:00:1793790968.694383   88364 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project'

In [5]:
@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_phase1_filepath: Path
    checkpoint_phase2_filepath: Path

In [6]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        create_directories([
            Path(config.checkpoint_phase1_filepath).parent,
            Path(config.checkpoint_phase2_filepath).parent,
            Path(config.tensorboard_root_log_dir),
        ])
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_phase1_filepath=Path(config.checkpoint_phase1_filepath),
            checkpoint_phase2_filepath=Path(config.checkpoint_phase2_filepath),
        )

In [7]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    def _checkpoint_path(self, phase: int) -> Path:
        return {1: self.config.checkpoint_phase1_filepath,
                2: self.config.checkpoint_phase2_filepath}[phase]

    def get_callbacks(self, phase: int, patience: int = 5) -> list:
        if phase not in (1, 2):
            raise ValueError("phase must be 1 or 2")
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_dir = Path(self.config.tensorboard_root_log_dir) / f"phase{phase}" / f"tb_logs_at_{timestamp}"
        return [
            tf.keras.callbacks.TensorBoard(log_dir=str(tb_dir)),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(self._checkpoint_path(phase)),
                monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True),   # the phase handoff file
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", mode="min",
                patience=patience, restore_best_weights=True),
        ]

In [8]:
try:
    config = ConfigurationManager()
    prepare_callbacks = PrepareCallback(config=config.get_prepare_callback_config())
    callback_list_p1 = prepare_callbacks.get_callbacks(phase=1)
except Exception as e:
    raise e

[2026-11-04 12:16:12,014: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-11-04 12:16:12,018: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-04 12:16:12,019: INFO: common: created directory at: artifacts]
[2026-11-04 12:16:12,021: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoint_dir]
[2026-11-04 12:16:12,022: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoint_dir]
[2026-11-04 12:16:12,023: INFO: common: created directory at: artifacts/prepare_callbacks/tensorboard_log_dir]
